In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
df = pd.read_csv('turtles.csv', delimiter='\t')

In [3]:
df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8861 entries, 0 to 8860
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   8861 non-null   int64  
 1   binomial_name        8812 non-null   str    
 2   registration number  8832 non-null   str    
 3   shell_length         8774 non-null   float64
 4   shell_width          8861 non-null   int64  
 5   head_length          8715 non-null   float64
 6   head_width           8715 non-null   float64
 7   flipper_length_1     8861 non-null   int64  
 8   flipper_width_1      8861 non-null   int64  
 9   flipper_length_2     8861 non-null   int64  
 10  flipper_width_2      8861 non-null   int64  
 11  flipper_length_3     8760 non-null   float64
 12  flipper_width_3      8760 non-null   float64
 13  flipper_length_4     8760 non-null   float64
 14  flipper_width_4      8760 non-null   float64
 15  circle_count         8861 non-null   int64  
 16 

In [4]:
# Списки признаков для предобработки
cat_features = ['binomial_name']
num_features = [
    'shell_length', 'shell_width', 'head_length', 'head_width',
    'flipper_length_1', 'flipper_width_1',
    'flipper_length_2', 'flipper_width_2',
    'flipper_length_3', 'flipper_width_3',
    'flipper_length_4', 'flipper_width_4',
    'circle_count', 'measure_count','weight'    
]
special_features = ['shell_crack']

In [5]:
# Создаем объекты SimpleImputer
cat_imputer = SimpleImputer(strategy='most_frequent')
num_imputer = SimpleImputer(strategy='median')
zero_imputer = SimpleImputer(strategy='constant', fill_value=0)

In [6]:
# Объедините всё в ColumnTransformer и примените трансформер к данным:
imputer_ct = ColumnTransformer(
    transformers=[
        ('cat_imputer', cat_imputer, cat_features),
        ('num_imputer', num_imputer, num_features),
        ('zero_imputer', zero_imputer, special_features)
    ]
)

features_filled = imputer_ct.fit_transform(df)
df_filled = pd.DataFrame(features_filled, columns=df.columns)

ValueError: Cannot use median strategy with non-numeric data:
could not convert string to float: '3,0'

In [ ]:
# Удалите строки с некорректным весом 
# Выведите размер датасета до и после удаления
before_shape = df.shape
df = df[df['weight'] != 0].reset_index(drop=True)
after_shape = df.shape
print('before:', before_shape, 'after:', after_shape)

In [ ]:
def fix_scale(series, threshold):
    return np.where(series > threshold, series / 10, series)

test_series = np.array([500, 2600, 10000, 200])
result = fix_scale(test_series, 2500)
print(result)

In [ ]:
# Допишите списки признаков для каждой группы
shell_cols = ['shell_length', 'shell_width']
head_cols = ['head_length', 'head_width']
flipper_cols = [
    'flipper_length_1',
    'flipper_width_1',
    'flipper_length_2',
    'flipper_width_2', 
    'flipper_length_3',
    'flipper_width_3',
    'flipper_length_4', 
    'flipper_width_4'
]

for columns, threshold in [(shell_cols, 2500), (head_cols, 400), (flipper_cols, 1000)]:
    for col in columns:
        df[col] = np.where(df[col] > threshold, df[col] / 10, df[col])

In [ ]:
# Посчитайте число уникальных значений до и после приведения к нижнему регистру
uniq_before = df['binomial_name'].nunique()
df['binomial_name'] = df['binomial_name'].str.strip().str.lower()
uniq_after = df['binomial_name'].nunique()
print('unique before:', uniq_before, 'unique after:', uniq_after)

In [ ]:
# Выберите и обучите кодировщик в зависимости от числа уникальных значений
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder_columns = encoder.fit_transform(df[['binomial_name']])
encoder_df = pd.DataFrame(
    encoder_columns,
    columns=encoder.get_feature_names_out(['binomial_name'])
)

# Соберите итоговый датасет df_encoded:
df_encoded = pd.concat([df.drop(columns='binomial_name'), encoder_df], axis=1)
print(list(df_encoded.columns))

In [ ]:
num_features = [
    'shell_length', 'shell_width', 'head_length', 'head_width',
    'flipper_length_1', 'flipper_width_1',
    'flipper_length_2', 'flipper_width_2',
    'flipper_length_3', 'flipper_width_3',
    'flipper_length_4', 'flipper_width_4',
    'circle_count', 'measure_count'
]

# Создайте объект StandardScaler
scaler = StandardScaler()

# Примените scaler к датасету
df[num_features] = scaler.fit_transform(df[num_features])

print(df)

In [7]:
special_features = ['shell_crack', 'weight']

# Создайте пайплайны для категориальных и числовых признаков
cat_pipeline = Pipeline(
    steps=[
        ('cat_imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
    ]
)

num_pipeline = Pipeline(
    steps=[
        ('num_imputer', SimpleImputer(strategy='median')),
        ('stand_scaler', StandardScaler())
    ]
)

# Объедините все шаги в ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cat_pipeline', cat_pipeline, cat_features),
        ('num_pipeline', num_pipeline, num_features),
        ('simple_imputer', SimpleImputer(strategy='constant', fill_value=0), special_features)
    ]
)

# Примените транфсормер к датасету
transformed_features = preprocessor.fit_transform(df)
feature_names = preprocessor.get_feature_names_out()

df_transformed = pd.DataFrame(transformed_features, columns=feature_names)
print(df_transformed.head(5))

ValueError: Cannot use median strategy with non-numeric data:
could not convert string to float: '3,0'